In [ ]:
import os
import sys
# add path to custom functions
module_path = os.path.abspath(os.path.join('.')) #../..
if module_path not in sys.path:
    sys.path.append(module_path+"/scripts/py_functions")
# import custom functions
from map_plot_tools import *
from line_plot_tools import *
from colorbar_funcs import *
from data_funcs import *

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
#np.set_printoptions(threshold=np.inf) # disable truncation
import metpy.calc as mp
import pandas as pd 
from scipy.stats import pearsonr

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo
import seaborn as sns
# settings
%config InlineBackend.figure_format = 'retina'

# top level data directory (override with the WORK_DATA_DIR env var; see config/paths.env.example)
dpath0=os.environ.get('WORK_DATA_DIR', '/glade/work/dervlamk')
# save figs here
opath=os.environ.get('WORK_DATA_DIR', '/glade/work/dervlamk')

In [ ]:
def sigtest(yearmean1,yearmean2,timemean1,timemean2):
    ptvals = ttest_rel(yearmean1,yearmean2, axis=0)
    diff = timemean1-timemean2
    diff_mask = np.ma.masked_where(ptvals[1] > 0.1,diff)
    return diff, diff_mask, ptvals

# test with different sample sizes
def sigtest2n(yearmean1,yearmean2,timemean1,timemean2):
    ptvals = ttest_ind(yearmean1,yearmean2, axis=0, equal_var = False)
    diff = timemean1-timemean2
    diff_mask = np.ma.masked_where(ptvals[1] > 0.1,diff)
    return diff, diff_mask, ptvals

def windSpd(u,v):
   windSpd=np.sqrt(u**2 + v**2)
   return windSpd

In [ ]:
#=== SET FILE PATH INFO

files = {}

for sim in ['pi']:
    files[sim] = {}
    for varn in ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS']:
        files[sim][varn] = f'{dpath0}/PI/dh.precIsotopes.atm.iPI.nc'
    for varn in ['PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS']:
        files[sim][varn] = f'{dpath0}/PI/o.precIsotopes.atm.iPI.nc'
    for varn in ['PRECC', 'PRECL', 'TS', 'PSL']:
        files[sim][varn] = f'{dpath0}/PI/b.e12.B1850C5.f19_g16.iPI.01.400-499.climo.nc'
    for varn in ['Q','U','V','OMEGA', 'Z3']: # these are in their own files because I converted the pressure levels from hybrid to standard
        files[sim][varn] = f'{dpath0}/PI/{varn}.iPI.0400-0499.climo.nc'

for sim in ['lgm']:
    files[sim] = {}
    for varn in ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS',
                'PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS',
                'PRECC', 'PRECL', 'TS', 'PSL']:
        files[sim][varn] = f'{dpath0}/LGM/b.e12.B1850C5.f19_g16.i21ka.03.cam.h0.0801-0900.climo.nc'
    for varn in ['Q','U','V','OMEGA', 'Z3']: # these are in their own files because I converted the pressure levels from hybrid to standard
        files[sim][varn] = f'{dpath0}/LGM/{varn}.i21ka.0801-0900.climo.nc'


In [ ]:
#=== PROCESS DATA

dat={ }
sims=['pi','lgm']
varns = ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS', 
         'PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS', 
         'PRECC', 'PRECL','TS','U','V','OMEGA','Q','Z3','PSL']

for sim in ['pi']:
    dat[sim]={}
    for varn in varns:
        dat[sim][varn]=xr.open_dataset(files[sim][varn])[varn]
        dat[sim][varn].attrs['original_time_values'] = dat[sim][varn].time
        # update time axis?
        dat[sim][varn]=dat[sim][varn].rename({'time':'month'})
        dat[sim][varn]=dat[sim][varn].assign_coords(month=[1,2,3,4,5,6,7,8,9,10,11,12])
     
for sim in ['lgm']:
    dat[sim]={}
    for varn in varns:
        dat[sim][varn]=xr.open_dataset(files[sim][varn])[varn]
        dat[sim][varn].attrs['original_time_values'] = dat[sim][varn].time
        # update time axis?
        dat[sim][varn]=dat[sim][varn].rename({'time':'month'})
        dat[sim][varn]=dat[sim][varn].assign_coords(month=[1,2,3,4,5,6,7,8,9,10,11,12])

In [ ]:
# Gather values

dDp={}
d18Op={}
prec={}
precc={}
precl={}
sims=['pi','lgm']

for sim in sims:
    ptiny=1e-18;
    
    ## Precipitation
    precc[sim] = dat[sim]['PRECC']*1000*60*60*24
    precl[sim] = dat[sim]['PRECL']*1000*60*60*24
    # calculate total precip from convective and large-scale prec vars (snow+rain). convert from m/s to mm/day
    prec[sim] = (dat[sim]['PRECC'] + dat[sim]['PRECL'])*1000*60*60*24
    prec[sim].attrs['units'] = 'mm/day'
    prec[sim].attrs['long_name'] = 'total precipitation'
    prec[sim].attrs['source'] = 'PRECC + PRECL'
    # calculate precipitation weights by month
    annual_total_p = prec[sim].sum(dim="month")
    pWeights = prec[sim]/annual_total_p
    
    ## Hydrogen
    phyd = dat[sim]['PRECRC_H2Or'] + dat[sim]['PRECRL_H2OR'] + dat[sim]['PRECSC_H2Os'] + dat[sim]['PRECSL_H2OS']
    pdeu = dat[sim]['PRECRC_HDOr'] + dat[sim]['PRECRL_HDOR'] + dat[sim]['PRECSC_HDOs'] + dat[sim]['PRECSL_HDOS']
    # replace very small ph values with a tiny value
    phyd = phyd.where(phyd > ptiny, ptiny) 
    # turn into per mil notation
    dd = (pdeu/phyd - 1)*1000 
    # Multiply isotope values by weights
    dDp[sim] = dd*pWeights
    
    ## Oxygen
    p16o = dat[sim]['PRECRC_H216Or'] + dat[sim]['PRECRL_H216OR'] + dat[sim]['PRECSC_H216Os'] + dat[sim]['PRECSL_H216OS']
    p18o = dat[sim]['PRECRC_H218Or'] + dat[sim]['PRECRL_H218OR'] + dat[sim]['PRECSC_H218Os'] + dat[sim]['PRECSL_H218OS']
    # replace very small ph values with a tiny value
    p16o = p16o.where(p16o > ptiny, ptiny)
    # turn into per mil notation
    do = (p18o/p16o - 1)*1000 
    # Multiply isotope values by weights
    d18Op[sim] = do*pWeights

ts={}
for sim in ['pi','lgm']:
    ts[sim] = dat[sim]['TS']

In [ ]:
#=== Calculate LGM-PI differences

dDdiff = dDp['lgm'] - dDp['pi']
pdiff = prec['lgm'] - prec['pi']
pcdiff = precc['lgm'] - precc['pi']
pldiff = precl['lgm'] - precl['pi']
tsdiff = ts['lgm'] - ts['pi']
wdiff = dat['lgm']['OMEGA'] - dat['pi']['OMEGA']
udiff = dat['lgm']['U'] - dat['pi']['U']
vdiff = dat['lgm']['V'] - dat['pi']['V']

In [ ]:
#=== LOAD OTHER RELEVANT DATA

# IMERG precipitation
# BASELINE: 2001-2018 everywhere in this project. This notebook previously used 2011-2018,
# which disagreed with fig1_swna_modern_climate.ipynb (2001-2018) — two observational
# baselines in one paper. Standardised on 2001-2018 (Aug 2025).
# NOTE: the derived climo below has NOT been rebuilt yet. Un-comment the block and run it
# once on Casper (needs obs.imerg.precip.2001-2018.nc under $WORK_DATA_DIR/obs_data/)
# before this cell will load.
ds = xr.open_dataset('imerg.gn.2001-2018.climo.nc').precipitation
"""
# imerg precipitation
filen=f'{dpath0}/obs_data/obs.imerg.precip.2001-2018.nc'
ds = xr.open_dataset(filen).precipitation.transpose('time','lat','lon').groupby("time.month").mean(dim='time') * 24 # convert from mm/hr to mm/day
ds.attrs['units'] = 'mm/day'
ds.attrs['Units'] = 'mm/day'
# Convert to 0:360
nx   = len(ds.lon)
lons = np.linspace(0,360,nx)
ds['lon'] = lons
ds=ds.roll(lon=3600)
ds.attrs['source_filename'] = 'obs.imerg.precip.2001-2018.nc'
ds.to_netcdf('imerg.gn.2001-2018.climo.nc', mode='w')
"""

# ETOPO05 topography
filen = f'{dpath0}/obs_data/obs.etopo5.zsurf.nc'
etopo_full = xr.open_dataset(f'{filen}').ROSE
etopo = etopo_full.where(etopo_full>=0, np.nan) # remove bathymetry
etopoSWNA = etopo_full.sel(ETOPO05_X=slice(235,275), ETOPO05_Y=slice(10,42))

# load proxy timeslice mean values (updated as of Aug 2025)
# moved from proxy_data/ to data/processed/ (Aug 2025); path is relative to the repo root,
# which is where notebooks must be launched from
proxydD = pd.read_csv('data/processed/timeslice_mean_proxy_dDraw.csv')
proxydD

In [ ]:
"""
### +++ CALCULATE TIME-MEANS +++ ###

seasons=['DJF','JFM','JAS','JJAS']
seas_mean = { 'pi': {}, 'lgm': {} }
ann_seas_mean = { 'pi': {}, 'lgm': {} }

print('Calculating seasonal means for...')
for sim in ['pi','lgm']:
    print(sim)
    for var in ['d
    for season in seasons:
        months = get_season(season=season)
        # seasonal mean for whole timeseries
        custom_seasons = xr.where(dat[sim][var]['time'].dt.month.isin(mons), season, 'Other')
        seas_mean[key][run] = dat[key][run].groupby(custom_seasons).mean('time').rename({'month':'season'}).sel(season=season)
        # seasonal mean by year
        ann_seas_mean[key][run] = dat[key][run].sel(time=dat[key][run]['time'].dt.month.isin(mons)).groupby('time.year').mean(dim='time')

print('Done.')
"""

In [ ]:
"""
### +++ COMPARING ALL MODEL RUNS TO MODEL CTRL. +++ ###

# initialize dictionaries
model_diff      = { 'flor' : {} }
model_diff_mask = { 'flor' : {} }
model_ptvals    = { 'flor' : {} }

# names of modified topography runs
flor_mod_runs=['hicam','hitopo']

print('Significance testing for:')
for key in model_diff.keys():
    print(f'{key}')
    for run in flor_mod_runs:
        print(f'...{run}')
        # calculate significance of model - obs difference
        diff_, diff_mask_, ptvals_ = sigtest(ann_jas_mean[key][run], ann_jas_mean[key]['ctrl'],
                                             jas_mean[key][run], jas_mean[key]['ctrl'])
        model_diff[key][run] = diff_
        model_diff_mask[key][run] = diff_mask_
        model_ptvals[key][run] = ptvals_

print('Done.')
"""

# FIGS

## Composite Fig

### ANN, JFM, & JAS

In [ ]:
# Proxy Data (based on C30 handpicked data as of Aug 2025)
clons=proxydD['lon'].values
clats=proxydD['lat'].values
ddiff = proxydD['lgm_dD'].values - proxydD['holocene_dD'].values
# Model Data
lon = dat['pi']['PRECC'].lon
lat = dat['pi']['PRECC'].lat
# plot specs
lw=1
bbox={'boxstyle':'square','fc':'white','ec':'black','alpha':1,'pad':0.2}
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':12, 'ha':'center', 'va':'center', 'rotation':90}
titles = np.array([
    r'$\mathbf{\Delta \delta} \mathbf{D_{precip}}$',  # ΔδD_precip
    r'$\mathbf{\Delta}$Precipitation',                # ΔPrecipitation
    r'$\mathbf{\Delta \omega_{500}}$ and 850mb WIND'  # Δω_500 & 850mb WIND
])
seasons=np.array(['MEAN ANNUAL', 'JFM', 'JAS'])
letters=np.array(['A','B','C','D','E','F','G','H','I','J','K'])
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-120., -82.5, 10., 36.]
# vector specs
skip_n=1
w=0.0075
scalef=10
key_length=2
# isotopes cmap
icmap=cm.RdBu_r #cmo.balance
#icmap,_,_,_=get_settings(field='precip', diff=True)
ivmin=-3
ivmax=3
ilevels=np.linspace(ivmin, ivmax, 25)
inorm=mpl.colors.BoundaryNorm(ilevels, icmap.N)
# precip cmap
pcmap,_,_,_=get_settings(field='precip', diff=True)
pvmin=-2.5
pvmax=2.5
plevels=np.linspace(pvmin, pvmax, 21)
pnorm=mpl.colors.BoundaryNorm(plevels, pcmap.N)
# omega cmap
ocmap=cmo.balance
ovmin=-0.05
ovmax=0.05
olevels=np.linspace(ovmin, ovmax, 21)
onorm=mpl.colors.BoundaryNorm(olevels, ocmap.N)

# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=3, ncols=3, figsize=(15,10), subplot_kw={'projection': proj}, layout='constrained')
#fig.text(.5,1.0,'iCESM1.2 LGM (21ka) $-$ PI', size=16, weight='bold', ha='center')

for i in [0,1,2]:
    ax[i,0].text(map_bnds[0]-5, map_bnds[2]+((map_bnds[3]-map_bnds[2])/2), seasons[i], **text_kw2)
    ax[0,i].text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+1, titles[i], **text_kw1)

#dDp
cf1=ax[0,0].pcolormesh(lon, lat, dDdiff.mean(dim="month"), cmap=icmap, norm=inorm, transform=trans)
ax[1,0].pcolormesh(lon, lat, dDdiff[0:3,:,:].mean(dim="month"), cmap=icmap, norm=inorm, transform=trans)
ax[2,0].pcolormesh(lon, lat, dDdiff[6:9,:,:].mean(dim="month"), cmap=icmap, norm=inorm, transform=trans)
for i in [0,1,2]:
    ax[i,0].scatter(x=clons, y=clats, c=ddiff, cmap=icmap, norm=inorm, alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)
for i in [0,1]:
    ax[0,0].text(clons[i]-1.1, clats[i], f'{ddiff[i]:.1f}‰', fontsize=10, weight='bold', ha='right', bbox=bbox, zorder=100)
    
# precip 
cf2=ax[0,1].pcolormesh(lon, lat, pdiff.mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)
ax[1,1].pcolormesh(lon, lat, pdiff[0:3,:,:].mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)
ax[2,1].pcolormesh(lon, lat, pdiff[6:9,:,:].mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)

# omega and winds
cf3=ax[0,2].pcolormesh(lon, lat, wdiff.sel(lev_p=500.0).mean(dim="month"), cmap=ocmap, norm=onorm, transform=trans)
q1=ax[0,2].quiver(lon[::skip_n], lat[::skip_n], udiff.sel(lev_p=850.0).mean(dim="month")[::skip_n,::skip_n], vdiff.sel(lev_p=850.0).mean(dim="month")[::skip_n,::skip_n],
                  color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=100)
ax[1,2].pcolormesh(lon, lat, wdiff.sel(lev_p=500.0)[0:3,:,:].mean(dim="month"), cmap=ocmap, norm=onorm, transform=trans)
ax[1,2].quiver(lon[::skip_n], lat[::skip_n], udiff.sel(lev_p=850.0)[0:3,:,:].mean(dim="month")[::skip_n,::skip_n], vdiff.sel(lev_p=850.0)[0:3,:,:].mean(dim="month")[::skip_n,::skip_n],
               color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=100)
ax[2,2].pcolormesh(lon, lat, wdiff.sel(lev_p=500.0)[6:9,:,:].mean(dim="month"), cmap=ocmap, norm=onorm, transform=trans)
ax[2,2].quiver(lon[::skip_n], lat[::skip_n], udiff.sel(lev_p=850.0)[6:9,:,:].mean(dim="month")[::skip_n,::skip_n], vdiff.sel(lev_p=850.0)[6:9,:,:].mean(dim="month")[::skip_n,::skip_n],
               color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=100)

#ax[0,1].scatter(clons, clats, c='k', s=150, alpha=1, transform=trans, zorder=100)
qk=ax[0,2].quiverkey(q1, .95, 1.035, key_length, rf'{key_length} m/s', labelcolor='k', labelpos='W', fontproperties={'size':9})


for i,ax in enumerate(ax.flat):
    ax.contour(etopoSWNA.ETOPO05_X, etopoSWNA.ETOPO05_Y, etopoSWNA, 
           levels=np.linspace(800,3200,4), linewidths=0.5, colors='k', transform=trans)
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)
    #ax.add_feature(cfeature.STATES, linewidth=0.5)
    ax.text(map_bnds[0]-1, map_bnds[3]+0.5, letters[i], **text_kw1)
    ring=LinearRing(list(zip([-113., -105, -105, -113.], [18,  18,  33,  33])))
    ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax.set_extent(map_bnds, crs=trans)
    if i in [0,3]:
        gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False; gl.bottom_labels=False
    if i==6:
        gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
    if i in [7,8]:
        gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False; gl.left_labels=False
    else:
        gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=False)


cbar_ax1 = fig.add_axes([0.05, -0.025, 0.275, 0.02])
cbar1 = fig.colorbar(cf1, ticks=[-3,-2,-1,0,1,2,3], orientation='horizontal', extend='both', cax=cbar_ax1)
cbar1.set_label(u'[‰]', weight='normal', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

cbar_ax2 = fig.add_axes([0.375, -0.025, 0.275, 0.02])
cbar2 = fig.colorbar(cf2, ticks=[-2,-1,0,1,2], orientation='horizontal', extend='both', cax=cbar_ax2)
cbar2.set_label('[mm day$^{-1}$]', weight='normal', labelpad=5, rotation=0)
cbar2.ax.tick_params(labelsize=10)
for tick in cbar2.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

cbar_ax3 = fig.add_axes([0.7, -0.025, 0.275, 0.02])
cbar3 = fig.colorbar(cf3, ticks=[-0.04, -0.02, 0, 0.02, 0.04], orientation='horizontal', extend='both', cax=cbar_ax3)
cbar3.set_label('[Pa s$^{-1}$]', weight='normal', labelpad=5, rotation=0)
cbar3.ax.tick_params(labelsize=10)
for tick in cbar3.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

#plt.savefig("cesm1.2_LGM-PI_diffs.pdf", bbox_inches='tight')

### JAS only

In [ ]:
# Proxy Data (based on C30 handpicked data as of Aug 2025)
clons=proxydD['lon'].values
clats=proxydD['lat'].values
ddiff = proxydD['lgm_dD'].values - proxydD['holocene_dD'].values
# Model Data
lon = dat['pi']['PRECC'].lon
lat = dat['pi']['PRECC'].lat
im=5 # June
em=9 # Sept
# plot specs
lw=1
bbox={'boxstyle':'square','fc':'white','ec':'black','alpha':1,'pad':0.2}
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':12, 'ha':'center', 'va':'center', 'rotation':90}
titles = np.array([
    r'$\mathbf{\Delta \delta} \mathbf{D_{precip}}$',  # ΔδD_precip
    r'$\mathbf{\Delta}$Precipitation',                # ΔPrecipitation
    r'$\mathbf{\Delta \omega_{500}}$ and 850mb WIND'  # Δω_500 & 850mb WIND
])
months=['January-February-March', 'June-July-August-September', 'July-August-September']
if im==0:
    t_months=months[0]
elif im==5:
    t_months=months[1]
elif im==6:
    t_months=months[2]
else:
    t_months='<not_defined>'
letters=np.array(['A','B','C','D','E','F','G','H','I','J','K'])
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-120., -82.5, 10., 36.]
# vector specs
skip_n=1
w=0.0075
scalef=10
key_length=2
# isotopes cmap
icmap=cm.RdBu_r #cmo.balance
#icmap,_,_,_=get_settings(field='precip', diff=True)
ivmin=-3
ivmax=3
ilevels=np.linspace(ivmin, ivmax, 25)
inorm=mpl.colors.BoundaryNorm(ilevels, icmap.N)
# precip cmap
pcmap,_,_,_=get_settings(field='precip', diff=True)
pvmin=-2.5
pvmax=2.5
plevels=np.linspace(pvmin, pvmax, 21)
pnorm=mpl.colors.BoundaryNorm(plevels, pcmap.N)
# omega cmap
ocmap=cmo.balance
ovmin=-0.05
ovmax=0.05
olevels=np.linspace(ovmin, ovmax, 21)
onorm=mpl.colors.BoundaryNorm(olevels, ocmap.N)

# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(16,4.5), subplot_kw={'projection': proj}, layout='constrained')
fig.text(.5,1,t_months+' iCESM1.2 LGM (21ka) $-$ PI', **text_kw)

for i in [0,1,2]:
    ax[i].text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+1, titles[i], **text_kw1)

#dDp
cf1=ax[0].pcolormesh(lon, lat, dDdiff[im:em,:,:].mean(dim="month"), cmap=icmap, norm=inorm, transform=trans)
ax[0].scatter(x=clons, y=clats, c=ddiff, cmap=icmap, norm=inorm, alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)
for i in [0,1]:
    ax[0].text(clons[i]-1.1, clats[i], f'{ddiff[i]:.1f}‰', fontsize=10, weight='bold', ha='right', bbox=bbox, zorder=100)

# precip 
cf2=ax[1].pcolormesh(lon, lat, pdiff[im:em,:,:].mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)
ax[1].scatter(clons, clats, ec='k', fc='w', s=150, alpha=1, transform=trans, zorder=100)

# omega and winds
cf3=ax[2].pcolormesh(lon, lat, wdiff.sel(lev_p=500.0)[im:em,:,:].mean(dim="month"), cmap=ocmap, norm=onorm, transform=trans)
q1=ax[2].quiver(lon[::skip_n], lat[::skip_n],
                udiff.sel(lev_p=850.0)[im:em,:,:].mean(dim="month")[::skip_n,::skip_n],
                vdiff.sel(lev_p=850.0)[im:em,:,:].mean(dim="month")[::skip_n,::skip_n],
               color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=100)
qk=ax[2].quiverkey(q1, .95, 1.035, key_length, rf'{key_length} m/s', labelcolor='k', labelpos='W', fontproperties={'size':9})
ax[2].scatter(clons, clats, ec='k', fc='w', s=150, alpha=1, transform=trans, zorder=100)


for i,ax in enumerate(ax.flat):
    ax.contour(etopoSWNA.ETOPO05_X, etopoSWNA.ETOPO05_Y, etopoSWNA, 
           levels=np.linspace(800,3200,4), linewidths=0.5, colors='k', transform=trans)
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)
    #ax.add_feature(cfeature.STATES, linewidth=0.5)
    ax.text(map_bnds[0]-1, map_bnds[3]+0.5, letters[i], **text_kw1)
    ring=LinearRing(list(zip([-113., -105, -105, -113.], [18,  18,  33,  33])))
    ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax.set_extent(map_bnds, crs=trans)
    if i in [0]:
        gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False; gl.bottom_labels=True
    if i in [1,2]:
        gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False; gl.left_labels=False


cbar_ax1 = fig.add_axes([0.05, 0, 0.275, 0.05])
cbar1 = fig.colorbar(cf1, ticks=[-3,-2,-1,0,1,2,3], orientation='horizontal', extend='both', cax=cbar_ax1)
cbar1.set_label(u'[‰]', weight='normal', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

cbar_ax2 = fig.add_axes([0.375, 0, 0.275, 0.05])
cbar2 = fig.colorbar(cf2, ticks=[-2,-1,0,1,2], orientation='horizontal', extend='both', cax=cbar_ax2)
cbar2.set_label('[mm day$^{-1}$]', weight='normal', labelpad=5, rotation=0)
cbar2.ax.tick_params(labelsize=10)
for tick in cbar2.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

cbar_ax3 = fig.add_axes([0.7, 0, 0.275, 0.05])
cbar3 = fig.colorbar(cf3, ticks=[-0.04, -0.02, 0, 0.02, 0.04], orientation='horizontal', extend='both', cax=cbar_ax3)
cbar3.set_label('[Pa s$^{-1}$]', weight='normal', labelpad=5, rotation=0)
cbar3.ax.tick_params(labelsize=10)
for tick in cbar3.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

#plt.savefig("cesm1.2_LGM-PI_diffs.pdf", bbox_inches='tight')

## Precip

### Convective vs. Large-Scale Precip

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':24, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'center', 'rotation':90}
titles=np.array([r'PRECC', r'PRECL'])
seasons=np.array(['ANN','JFM', 'JAS'])
letters=['A','B','C','D','E','F']
tx=-101.75
ty=36
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-120., -82.5, 10., 36.]
# precip cmap
pcmap,_,_,_=get_settings(field='precip', diff=True)
pvmin=-2.5
pvmax=2.5
plevels=np.linspace(pvmin, pvmax, 21)
pnorm=mpl.colors.BoundaryNorm(plevels, pcmap.N)

# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(20,10), layout='constrained', subplot_kw={'projection':proj})

for i in [0,1,2]:
    ax[0,i].text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+1, seasons[i], **text_kw)
    if 0 <= i <= 1:
        ax[i,0].text(map_bnds[0]-7, map_bnds[2]+((map_bnds[3]-map_bnds[2])/2), titles[i], **text_kw2)
    else:
        pass

ax[0,0].pcolormesh(pcdiff.lon, pcdiff.lat, pcdiff.mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)
ax[0,1].pcolormesh(pcdiff.lon, pcdiff.lat, pcdiff[0:3,:,:].mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)
ax[0,2].pcolormesh(pcdiff.lon, pcdiff.lat, pcdiff[6:9,:,:].mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)

ax[1,0].pcolormesh(pcdiff.lon, pcdiff.lat, pldiff.mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)
ax[1,1].pcolormesh(pcdiff.lon, pcdiff.lat, pldiff[0:3,:,:].mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)
cf=ax[1,2].pcolormesh(pcdiff.lon, pcdiff.lat, pldiff[6:9,:,:].mean(dim="month"), cmap=pcmap, norm=pnorm, transform=trans)

for i, ax in enumerate(ax.flat): 
    # add box around core NAM domain
    lon_bnds = np.array([-113, -104, -104, -113])
    lat_bnds = np.array([20,  20,  34,  34])
    ring=LinearRing(list(zip(lon_bnds, lat_bnds)))
    #ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=3, linestyle='--', zorder=11) 
    # subplot labels
    ax.text(-121, ty+1, letters[i], **text_kw1)
    # map properties
    ax.coastlines(color='k', linewidth=2)
    ax.add_feature(cfeature.STATES, edgecolor='k', linewidth=2)
    ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=2)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# colorbar
cax=fig.add_axes([1.01, 0.15, 0.025, 0.7])
cbar=fig.colorbar(cf, ticks=[-3,-2,-1,0,1,2,3], orientation='vertical', extend='both', cax=cax)
cbar.set_label('$\Delta$ PRECIPITATION [mm/day]', labelpad=25, rotation=270, size=18, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=18)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

plt.savefig("cesm1.2_LGM-PI_precc_precl_diffs.pdf", bbox_inches='tight')

## Moisture Transport

In [ ]:
#=== Calculate moisture transport

qv = { 'pi':{}, 'lgm':{} }
qu = { 'pi':{}, 'lgm':{} }
mt = { 'pi':{}, 'lgm':{} }

for key in ['pi','lgm']:
    qu[key] = dat[key]['U']*dat[key]['Q']
    qv[key] = dat[key]['V']*dat[key]['Q']
    mt[key] = windSpd(qu[key], qv[key]) 

In [ ]:
# Model Data
lon = dat['pi']['Q'].lon
lat = dat['pi']['Q'].lat
# var specs
im=6
em=9
level=850
months=['January-February-March', 'June-July-August-September', 'July-August-September']
if im==0:
    t_months=months[0]
elif im==5:
    t_months=months[1]
elif im==6:
    t_months=months[2]
else:
    t_months='<not_defined>'
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'left', 'va':'bottom'}
titles=np.array(['PI', 'LGM', 'LGM$-$PI'])
# vector specs
skip_n=1
w=0.0075
scalef=.1
key_length=2
# isotopes cmap
cmap,_,_,_=get_settings(field='q', diff=False)
vmin=0
vmax=.1
levels=np.linspace(vmin, vmax, 21) 
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
# precip cmap
dcmap,_,_,_=get_settings(field='q', diff=True)
dvmin=-.03
dvmax=.03
dlevels=np.linspace(dvmin, dvmax, 19)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-135., -65., 10., 42.]


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20,5), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5,1,t_months+f' {level} mb MOISTURE TRANSPORT', **text_kw)

# Climo
cf1=ax[0].pcolormesh(lon, lat, mt['pi'][im:em].sel(lev_p=level).mean(dim='month'), cmap=cmap, norm=norm, transform=trans)
ax[1].pcolormesh(lon, lat, mt['lgm'][im:em].sel(lev_p=level).mean(dim='month'), cmap=cmap, norm=norm, transform=trans)
cf2=ax[2].pcolormesh(lon, lat, (mt['lgm'][im:em]-mt['pi'][im:em]).sel(lev_p=level).mean(dim='month'), cmap=dcmap, norm=dnorm, transform=trans)
q1=ax[2].quiver(lon[::skip_n], lat[::skip_n],
                (qu['lgm'][im:em]-qu['pi'][im:em]).sel(lev_p=level).mean(dim="month")[::skip_n,::skip_n],
                (qv['lgm'][im:em]-qv['pi'][im:em]).sel(lev_p=level).mean(dim="month")[::skip_n,::skip_n],
               color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=100)
qk=ax[2].quiverkey(q1, .95, 1.035, key_length, rf'{key_length} m/s', labelcolor='k', labelpos='W', fontproperties={'size':9})
ax[2].scatter(clons, clats, ec='k', fc='w', s=150, alpha=1, transform=trans, zorder=100)

for i in [0,1,2]:
    ax[i].contour(etopoSWNA.ETOPO05_X, etopoSWNA.ETOPO05_Y, etopoSWNA, 
           levels=np.linspace(1600,5600,13), linewidths=0.25, colors='k', transform=trans)
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].add_feature(cfeature.STATES, linewidth=0.5)
    ax[i].text(map_bnds[0], map_bnds[3]+0.5, titles[i], **text_kw1)
    ring=LinearRing(list(zip([-113., -104, -104, -113.], [18,  18,  33,  33])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
    if i in [1,2]:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False

cbar_ax1 = fig.add_axes([0.05, -0.025, 0.45, 0.05])
cbar1 = fig.colorbar(cf1, ticks=np.arange(0.005,.5,.01), orientation='horizontal', extend='max', cax=cbar_ax1)
cbar1.set_label(u'kg/kg/m/s', weight='normal', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

cbar_ax2 = fig.add_axes([0.535, -0.025, 0.45, 0.05])
cbar2 = fig.colorbar(cf2, ticks=np.arange(-.06,.06,.02), orientation='horizontal', extend='both', cax=cbar_ax2)
cbar2.set_label('$\Delta$ kg/kg/m/s', weight='normal', labelpad=5, rotation=0)
cbar2.ax.tick_params(labelsize=10)
for tick in cbar2.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

fig.text(0,-0.175, r'For iCESM1.2 LGM output.')
#plt.savefig("cesm1.2_LGM-PI_jjas_dDp_precip.pdf")

## Surface Pressure

In [ ]:
# Model Data
lon = dat['pi']['PSL'].lon
lat = dat['pi']['PSL'].lat
# var specs
im=5
em=9
level=850
months=['January-February-March', 'June-July-August-September', 'July-August-September']
if im==0:
    t_months=months[0]
elif im==5:
    t_months=months[1]
elif im==6:
    t_months=months[2]
else:
    t_months='<not_defined>'
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'left', 'va':'bottom'}
titles=np.array(['PI', 'LGM', 'LGM$-$PI'])
# isotopes cmap
cmap,_,_,_=get_settings(field='slp', diff=False)
vmin=1000
vmax=1040
levels=np.linspace(vmin, vmax, 21) 
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
# precip cmap
dcmap,_,_,_=get_settings(field='slp', diff=True)
dvmin=-15
dvmax=15
dlevels=np.linspace(dvmin, dvmax, 31)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-135., -65., 10., 42.]


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20,5), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5,1,t_months+' Surface Pressure', **text_kw)

# Climo
cf1=ax[0].pcolormesh(lon, lat, dat['pi']['PSL'][im:em].mean(dim='month')/100, cmap=cmap, norm=norm, transform=trans)
ax[1].pcolormesh(lon, lat, dat['lgm']['PSL'][im:em].mean(dim='month')/100, cmap=cmap, norm=norm, transform=trans)
cf2=ax[2].pcolormesh(lon, lat, (dat['lgm']['PSL'][im:em]-dat['pi']['PSL'][im:em]).mean(dim='month')/100, cmap=dcmap, norm=dnorm, transform=trans)
ax[2].scatter(clons, clats, ec='k', fc='w', s=150, alpha=1, transform=trans, zorder=100)

for i in [0,1,2]:
    ax[i].contour(etopoSWNA.ETOPO05_X, etopoSWNA.ETOPO05_Y, etopoSWNA, 
           levels=np.linspace(1600,5600,13), linewidths=0.25, colors='k', transform=trans)
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].add_feature(cfeature.STATES, linewidth=0.5)
    ax[i].text(map_bnds[0], map_bnds[3]+0.5, titles[i], **text_kw1)
    ring=LinearRing(list(zip([-113., -104, -104, -113.], [18,  18,  33,  33])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
    if i in [1,2]:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False

cbar_ax1 = fig.add_axes([0.05, -0.025, 0.45, 0.05])
cbar1 = fig.colorbar(cf1, ticks=np.arange(900,1100,5), orientation='horizontal', extend='max', cax=cbar_ax1)
cbar1.set_label(u'[mb]', weight='normal', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

cbar_ax2 = fig.add_axes([0.535, -0.025, 0.45, 0.05])
cbar2 = fig.colorbar(cf2, ticks=np.arange(-100,100,20), orientation='horizontal', extend='both', cax=cbar_ax2)
cbar2.set_label('$\Delta$[mb]', weight='normal', labelpad=5, rotation=0)
cbar2.ax.tick_params(labelsize=10)
for tick in cbar2.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

fig.text(0,-0.175, r'For iCESM1.2 LGM output.')
#plt.savefig("cesm1.2_LGM-PI_jjas_dDp_precip.pdf")

## Surface Temperature

In [ ]:
# Model Data
lon = dat['pi']['TS'].lon
lat = dat['pi']['TS'].lat
# var specs
im=5 #start month
em=9 #end month
months=['January-February-March', 'June-July-August-September', 'July-August-September']
if im==0:
    t_months=months[0]
elif im==5:
    t_months=months[1]
elif im==6:
    t_months=months[2]
else:
    t_months='<not_defined>'
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'left', 'va':'bottom'}
titles=np.array(['PI', 'LGM', 'LGM$-$PI'])
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-135., -65., 10., 42.]
# isotopes cmap
cmap,_,_,_=get_settings(field='temp', diff=False)
vmin=10
vmax=34
levels=np.linspace(vmin, vmax, 25) 
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
# precip cmap
dcmap,_,_,_=get_settings(field='temp', diff=True)
dvmin=-8
dvmax=8
dlevels=np.linspace(dvmin, dvmax, 25)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20,5), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5,1,t_months+' SURFACE TEMPERATURE', **text_kw)

# Climo
cf1=ax[0].pcolormesh(lon, lat, ts['pi'][im:em].mean(dim='month')-273.15, cmap=cmap, norm=norm, transform=trans)
ax[1].pcolormesh(lon, lat, ts['lgm'][im:em].mean(dim='month')-273.15, cmap=cmap, norm=norm, transform=trans)
cf2=ax[2].pcolormesh(lon, lat, tsdiff[im:em].mean(dim='month'), cmap=dcmap, norm=dnorm, transform=trans)

for i in [0,1,2]:
    ax[i].contour(etopoSWNA.ETOPO05_X, etopoSWNA.ETOPO05_Y, etopoSWNA, 
           levels=np.linspace(1600,5600,13), linewidths=0.25, colors='k', transform=trans)
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].add_feature(cfeature.STATES, linewidth=0.5)
    ax[i].text(map_bnds[0], map_bnds[3]+0.5, titles[i], **text_kw1)
    ring=LinearRing(list(zip([-113., -104, -104, -113.], [18,  18,  33,  33])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
    if i in [1,2]:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False

cbar_ax1 = fig.add_axes([0.05, -0.025, 0.45, 0.05])
cbar1 = fig.colorbar(cf1, ticks=np.arange(0,40,5), orientation='horizontal', extend='both', cax=cbar_ax1)
cbar1.set_label(u'[°C]', weight='normal', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

cbar_ax2 = fig.add_axes([0.535, -0.025, 0.45, 0.05])
cbar2 = fig.colorbar(cf2, ticks=np.arange(-8,8,2), orientation='horizontal', extend='both', cax=cbar_ax2)
cbar2.set_label('$\Delta$[°C]', weight='normal', labelpad=5, rotation=0)
cbar2.ax.tick_params(labelsize=10)
for tick in cbar2.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

fig.text(0,-0.175, r'For iCESM1.2 LGM output.')
#plt.savefig("cesm1.2_LGM-PI_jjas_dDp_precip.pdf")